In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pip

##  -U is for upgrade
##  -qqq is for quiet

!pip install -U -qqq git+https://github.com/qdosgit4/30040-experiments.git#subdirectory=experiment_5_reparameterisation_reimplementation

##  !pip show -f reparam

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [18]:
import argparse, time, os
from datetime import datetime

import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset, Subset
from torchvision import datasets
from torchvision.transforms import ToTensor

import matplotlib.pyplot as plt

from reparam.linear_bayesian import Linear_bayesian
from reparam.mini_model_reparam import Linear_model_reparam
from reparam.pi_dataset import Pi_dataset
##  from reparam.mini_model_deterministic import Linear_model_deterministic
##  from reparam.experiment_training_lib import run_training_loop
##  from reparam.experiment_utilise_lib import run_utilisation_loop_once, run_utilisation_loop_batch

print(torch.__version__)

2.10.0+cu128


In [ ]:
##import pkgutil

##for module in pkgutil.iter_modules():
##  print(module.name)

In [4]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"

##  Following is essential for reparameterisation;
##    bfloat16 can cause vanishing gradient.
torch.set_default_dtype(torch.float32)

In [15]:
class Linear_model_deterministic(nn.Module):

    def __init__(self, n: int, udist: tuple[int, int], random_seed: int):

        super().__init__()

        self.linear_relu_stack = nn.Sequential(
            nn.Linear(1, n),
            nn.ReLU(),
            nn.Linear(n, n),
            nn.ReLU(),
            nn.Linear(n, 1)
        )

        nn.init.uniform_(self.linear_relu_stack[0].weight, a=-udist[0], b=udist[1])
        nn.init.uniform_(self.linear_relu_stack[2].weight, a=-udist[0], b=udist[1])
        nn.init.uniform_(self.linear_relu_stack[4].weight, a=-udist[0], b=udist[1])


    def forward(self, x: torch.Tensor):

        logits = self.linear_relu_stack(x)

        out = torch.sigmoid(logits)

        return out, 0

In [16]:
def run_model(env):

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

    ##  Setup filename to store params to.

    filename = f"model_params_batch_{env['batch_n']}_epochs_{env['training_epochs']}_{env['params_name']}"

    file_path = os.path.join(*["/content", "drive", "MyDrive", filename])

    print(file_path)


    ##  Define model.

    if env["udist"] == (0, 0)

        model = Linear_model_reparam(
                    env["neurons_n"],
                    env["mu_w_init"],
                    env["mu_b_init"],
                    env["rho_w_init"],
                    env["rho_b_init"],
        ).to(device)

    else:

        model = Linear_model_deterministic(
                    env["neurons_n"],
                    env["udist"],
                    env["random_seed"],
        ).to(device)

    ##  Ensure multiple GPUs used if available.

    if torch.cuda.device_count() > 1:

        print(f"Using {torch.cuda.device_count()} GPUs!")

        model = nn.DataParallel(model, device_ids=[0, 1])

    ##  Either load params or load training data.

    print(env)

    if env["params_load"] is not None:

        run_utilisation_loop_once(
                    model,
                    env["params_load"],
                    env["params_load"].split('/')[-1] + "_plot",
                    env["sample_quantity"]
        )

    elif env["params_load_batch"] is not None:

        run_utilisation_loop_batch(
                    model,
                    env["params_load_batch"],
                    env["sample_quantity"]
        )

    else:

        train_dl = DataLoader(
                    Subset(
                        Pi_dataset("/content/drive/MyDrive/modules_work/COMP30040/datasets/pi_dataset_40000.txt"),
                        range(40000)
                    ),
                    batch_size = env["batch_n"],
                    shuffle=True
        )

        test_dl = DataLoader(
                    Subset(
                        Pi_dataset("/content/drive/MyDrive/modules_work/COMP30040/datasets/pi_dataset_8000.txt"),
                        range(8000)
                    ),
                    batch_size = env["batch_n"],
                    shuffle=True
        )

        run_training_loop(model,
                          train_dl,
                          test_dl,
                          env["training_epochs"],
                          file_path)
                          ##env["kl_loss_ratio"])


In [29]:
def run_training_loop(model: nn.Module, train_dl: DataLoader, test_dl: DataLoader, epochs: int, filename: str):

    ##  Define loss.

    loss = nn.BCELoss()

    ##  Define parameter optimisation mechanism.

    optimizer = torch.optim.SGD(model.parameters(), lr=1e-3)

    # while time.time() - start_time < 300:

    ##  Train either for fixed period of time or fixed quantity of epochs.

    start_time = time.time()
    ##  Output training start time.

    print(datetime.now().strftime("%H:%M:%S"))

    ##  Run train-test sequence.

    for t in range(epochs):

        print(f"Training...")

        train(train_dl, model, loss, optimizer)

        print(f"Testing...")

        test(test_dl, model, loss)

    ##  Output training time.

    print(f"{time.time() - start_time}s")

    # print(model.state_dict())

    ##  Save model params to plaintext.

    with open("model_params.post", "w") as f:

        state_dict = model.module.state_dict() if isinstance(model, nn.DataParallel) else model.state_dict()

        f.write(str(state_dict))

    ##  Generate timestamp and store weights for later loading back.

    print(filename)

    torch.save(model.state_dict(), filename)


def train(dl: DataLoader, model: nn.Module, loss: nn.Module,
          optimizer: torch.optim.Optimizer):

    ##  Put model into training mode; ensures gradient tracking.

    model.train()

    for batch, (X, y) in enumerate(dl):

        X, y = X.to(device), y.to(device)

        ##  Run X through model, generate prediction.

        y_hat, _ = model(X)

        ##  Calculate error of prediction.

        loss_res = loss(y_hat, y)


        ##  compute gradients numerically via backpropagation, back to
        ##  leaf nodes of graph.

        loss_res.backward()

        ##  Iterate parameters.

        optimizer.step()

        ##  Reset gradients within graph.

        optimizer.zero_grad()

        # optimizer.debug_off()

        if batch % 1000 == 0:

            pass

            # print(torch.stack([X, y, y_hat], dim=0).T)

            ##  It is not necessary to know the exact parameter
            ##  values, just that they are changing.

            # optimizer.debug_off()

            # print(model.state_dict().keys())

            # print(model.state_dict()['linear_relu_stack.0.weight_mu'].sum(),
            #       model.state_dict()['linear_relu_stack.0.rho_weight'].sum(),
            #       model.state_dict()['linear_relu_stack.0.mu_bias'].sum(),
            #       model.state_dict()['linear_relu_stack.0.rho_bias'].sum(),
            #       model.state_dict()['linear_relu_stack.2.weight_mu'].sum(),
            #       model.state_dict()['linear_relu_stack.2.rho_weight'].sum(),
            #       model.state_dict()['linear_relu_stack.2.mu_bias'].sum(),
            #       model.state_dict()['linear_relu_stack.2.rho_bias'].sum(),
            #       model.state_dict()['linear_relu_stack.4.weight_mu'].sum(),
            #       model.state_dict()['linear_relu_stack.4.rho_weight'].sum(),
            #       model.state_dict()['linear_relu_stack.4.mu_bias'].sum(),
            #       model.state_dict()['linear_relu_stack.4.rho_bias'].sum(),
            #       )

            # print(model.state_dict()['linear_relu_stack.0.weight_mu'][0])
            # print(model.state_dict()['linear_relu_stack.0.weight_mu'][0].grad)

            # print(torch.round(y_hat), y)

            # print(loss_res.item())

            # current = (batch + 1) * len(X)

            # print(f"[{current:>5d}/{len(train_dl.dataset):>5d}]")

            # print(f"loss_1: {loss_1:>7f}  [{current:>5d}/{size:>5d}]")


def test(dl: DataLoader, model: nn.Module, loss: nn.Module):

    ##  Setup parameters for loss function.

    model.eval()

    test_loss, correct = 0, 0

    with torch.no_grad():

        for X, y in dl:

            X, y = X.to(device), y.to(device)

            y_hat, _ = model(X)

            test_loss += loss(y_hat, y).item()

            # correct += (y_hat.argmax(1) == y).type(torch.float).sum().item()

    test_loss /= len(dl)

    # correct /= len(dl.dataset)

    print(f"{test_loss:>8f}")

In [30]:
env_train_det = {
            "neurons_n": 1024,
            "batch_n": 64,
            "training_epochs": 50,
            "params_name": "det",       ##  Filename to store params to.
            "params_load": None,        ##  Filename to load specific set of params from.
            "params_load_batch": None,  ##  Filename to wildcard match a multiple sets of params.
            "random_seed": 239852,
            "mu_w_init": 0.0,
            "mu_b_init": 0.0,
            "rho_w_init": 0.0,
            "rho_b_init": 0.0,
            "kl_loss_ratio": 0.0,       ##  Ratio to consider KL-divergence loss during sum of loss functions.
            "udist": (0.1, 0.1)         ##  Uniform distribution params, in the case of deterministic model.
}

env_train_reparam = {
            "neurons_n": 1024,
            "batch_n": 1,
            "training_epochs": 10,
            "params_name": "test",      ##  Filename to store params to.
            "params_load": None,        ##  Filename to load specific set of params from.
            "params_load_batch": None,  ##  Filename to wildcard match a multiple sets of params.
            "random_seed": 239852,
            "mu_w_init": 0.1,
            "mu_b_init": 0.1,
            "rho_w_init": -10.0,
            "rho_b_init": -10.0,
            "kl_loss_ratio": 0.0,       ##  Ratio to consider KL-divergence loss during sum of loss functions.
            "udist": (0, 0)             ##  Uniform distribution params, if not (0, 0) will trigger deterministic model.
}

#for k, v in env.items():
#            print(f"{k}: {v}")

torch.manual_seed(env_train["random_seed"])

#run_model(env_train_reparam)

run_model(env_train_det)

/content/drive/MyDrive/model_params_batch_64_epochs_50_det
{'neurons_n': 1024, 'batch_n': 64, 'training_epochs': 50, 'params_name': 'det', 'params_load': None, 'params_load_batch': None, 'random_seed': 239852, 'mu_w_init': 0.0, 'mu_b_init': 0.0, 'rho_w_init': 0.0, 'rho_b_init': 0.0, 'kl_loss_ratio': 0.0, 'udist': (0.1, 0.1)}
40000
8000
11:42:55
Training...
Testing...
0.646806
Training...
Testing...
0.625750
Training...
Testing...
0.605909
Training...
Testing...
0.589508
Training...
Testing...
0.569260
Training...
Testing...
0.546463
Training...
Testing...
0.524233
Training...
Testing...
0.501465
Training...
Testing...
0.478417
Training...
Testing...
0.454735
Training...
Testing...
0.432575
Training...
Testing...
0.411238
Training...
Testing...
0.390773
Training...
Testing...
0.372019
Training...
Testing...
0.352639
Training...
Testing...
0.336739
Training...
Testing...
0.319819
Training...
Testing...
0.304676
Training...
Testing...
0.291006
Training...
Testing...
0.279590
Training...
T

In [26]:
##  Note that this cell is an unedited dump of experiment_utilise_lib.py

from pathlib import Path
import lzma
import io

import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset, Subset
from torchvision import datasets
from torchvision.transforms import ToTensor

import matplotlib.pyplot as plt


torch.set_default_dtype(torch.float32)

device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"


def run_utilisation_loop_batch(model: nn.Module, batch_name: str, sample_quantity: int):

    print("loop batch")

    directory = Path('./params/')

    # Find all files with batch_name in filename

    for file in directory.glob(f"model_params*{batch_name}*xz"):

        print(str(file))

        if file.is_file():

            dist_metadata = str(file).split(batch_name)[-1].replace('.xz','')

            graph_filename = f"{batch_name}{dist_metadata}"

            params_path = f"./{str(file)}"

            run_utilisation_loop_once(model, params_path, graph_filename)

            # Do something with the file


def run_utilisation_loop_once(model: nn.Module, params_path: str, graph_filename: str, sample_quantity: int):

    ##  Decompress to memory.

    # with lzma.open(params_path, 'rb') as f:

    #     decompressed_data = f.read()

    ##  Load params.

    # checkpoint = torch.load(io.BytesIO(decompressed_data),

    checkpoint = torch.load(params_path,
                           weights_only = True,
                           map_location=torch.device('cpu')
                           )

    old_state_dict = checkpoint['state_dict'] if 'state_dict' in checkpoint else checkpoint

    print(old_state_dict)

    ##  Fix naming scheme and load module.

    model.load_state_dict(
                {k.replace('module.', '', 1): v for k, v in old_state_dict.items()}
    )

    model.eval()

    res = []

    interval = 1 / (2 ** 6)

    dl = DataLoader(
                TensorDataset(
                    torch.arange(1.57, 4.71 + interval, interval)
                ),
                batch_size = 1,
                shuffle = False
    )

    with torch.no_grad():

        # all_tensors = []

        # for i in range(1):

            outputs_set = []

            for batch, (X,) in enumerate(dl):

                X = X.to(device)

                # print(X)

                ##  Run X through model, generate prediction.

                y_hat, _ = model(X)

                print(X.item(), y_hat.item())

                outputs_set.append((X.item(), y_hat.squeeze(0).item()))

            # for batch, (X,) in enumerate(dl):

                # X = X.to(device)

                # # print(X)

                # ##  Run X through model, generate prediction.

                # sample_set = ()

                # for _ in range(sample_quantity):

                #     y_hat_sample, _ = model(X)

                #     sample_set = sample_set + (y_hat_sample,)

                # print(sample_set)

                # ##  Take mean of samples.

                # y_hat = torch.stack(sample_set).mean(dim=0)

                # print(f"p({X.item()}) = {y_hat.item()}")

                # ##  Store for later plotting

                # outputs_set = outputs_set + ((X.item(), y_hat.squeeze(0).item()),)

            # inner_tensor = torch.stack(outputs_set, dim=0)

            # all_tensors.append(inner_tensor)

            # print(all_tensors)

            x, y = zip(*outputs_set)

            plt.figure(figsize=(6, 4))

            plt.plot(
                        x, y,
                        marker='o',
                        markersize=4,          # small dots
                        markerfacecolor='black',
                        markeredgecolor='black',
                        linewidth=0.8,         # thin connecting line
                        color='black',
                        label='Trajectory'
            )

            # plt.scatter(
            #             x, y,
            #             color='black',          # point colour
            #             edgecolor='black',          # optional border around each marker
            #             s=10,                       # marker size
            #             label='Samples'
            # )

            plt.xlabel('Input value')
            plt.ylabel('Probability of π classification')
            plt.grid(True, ls='--', lw=0.5, alpha=0.7)
            plt.tight_layout()

            plt.tight_layout()

            plt.savefig(f"{graph_filename}.pdf", dpi=300)
            plt.close()

            return graph_filename



In [34]:
filename_reparam = f"/content/drive/MyDrive/model_params_batch_{env_train_reparam['batch_n']}_epochs_{env_train_reparam['training_epochs']}_{env_train_reparam['params_name']}",

env_load_reparam = {
            "params_name": None,
            "params_load": filename_reparam,
            "params_load_batch": None,
            "sample_quantity": 10
}

env_utilise_reparam = env_load_reparam | env_train_reparam

In [37]:
filename_det = f"/content/drive/MyDrive/model_params_batch_{env_train_det['batch_n']}_epochs_{env_train_det['training_epochs']}_{env_train_det['params_name']}"

env_load_det = {
            "params_name": None,
            "params_load": filename_det,
            "params_load_batch": None,
            "sample_quantity": 1
}

print(filename_det)

env_utilise_det = env_train_det | env_load_det

/content/drive/MyDrive/model_params_batch_64_epochs_50_det


In [38]:
for k, v in env_utilise_det.items():
            print(f"{k}: {v}")

run_model(env_utilise_det)

neurons_n: 1024
batch_n: 64
training_epochs: 50
params_name: None
params_load: /content/drive/MyDrive/model_params_batch_64_epochs_50_det
params_load_batch: None
random_seed: 239852
mu_w_init: 0.0
mu_b_init: 0.0
rho_w_init: 0.0
rho_b_init: 0.0
kl_loss_ratio: 0.0
udist: (0.1, 0.1)
sample_quantity: 1
/content/drive/MyDrive/model_params_batch_64_epochs_50_None
{'neurons_n': 1024, 'batch_n': 64, 'training_epochs': 50, 'params_name': None, 'params_load': '/content/drive/MyDrive/model_params_batch_64_epochs_50_det', 'params_load_batch': None, 'random_seed': 239852, 'mu_w_init': 0.0, 'mu_b_init': 0.0, 'rho_w_init': 0.0, 'rho_b_init': 0.0, 'kl_loss_ratio': 0.0, 'udist': (0.1, 0.1), 'sample_quantity': 1}
OrderedDict({'linear_relu_stack.0.weight': tensor([[-0.0081],
        [ 0.0292],
        [-0.1262],
        ...,
        [ 0.2240],
        [-0.0024],
        [-0.0939]]), 'linear_relu_stack.0.bias': tensor([-0.2420, -0.3562,  0.6145,  ...,  0.3541, -0.0713,  0.0679]), 'linear_relu_stack.2.weig